# Llama Email Classifier

This project builds a local AI email classifier using TinyLlama and `llama-cpp-python`. Emails are classified as **Priority**, **Updates**, or **Promotions**.

The final solution combines few-shot prompt engineering with a lightweight rule-based guardrail for obvious promotional language.

In [ ]:
!pip install llama-cpp-python==0.2.82 -q
!wget -q https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v0.3-GGUF/resolve/main/tinyllama-1.1b-chat-v0.3.Q4_K_M.gguf?download=true -O model.gguf

In [ ]:
import pandas as pd
from llama_cpp import Llama

emails_df = pd.read_csv('data/email_categories_data.csv')
emails_df.head()

In [ ]:
model_path = 'model.gguf'
llm = Llama(model_path=model_path)

In [ ]:
prompt = """
You are an email classifier.

Classify emails into exactly one category:

Priority:
Important or time-sensitive messages that require action or could cause
a negative consequence if ignored.

Updates:
Informational messages about an ongoing project, order, thread, or status
that do not require immediate action.

Promotions:
Advertising, sales, discounts, coupons, special offers, or shopping messages.

Return ONLY one word:
Priority
Updates
Promotions

Examples:

Q: Urgent: Server maintenance is required immediately.
A: Priority

Q: Your payment failed. Please update your billing information.
A: Priority

Q: Your package has shipped and will arrive tomorrow.
A: Updates

Q: The project has moved into testing. No action is required.
A: Updates

Q: Save 50% on our Spring Collection today!
A: Promotions

Q: Don't miss our biggest sale of the year.
A: Promotions
"""

In [ ]:
def process_message(email_content):
    email_lower = email_content.lower()

    promotion_words = [
        'sale', 'discount', '% off', 'coupon',
        'special offer', 'shop now', 'collection'
    ]

    if any(word in email_lower for word in promotion_words):
        return 'Promotions'

    input_prompt = prompt + '\n\nQ: ' + email_content + '\nA:'

    response = llm(
        input_prompt,
        max_tokens=5,
        temperature=0,
        stop=['Q:']
    )

    output = response['choices'][0]['text'].strip()

    if 'Promotions' in output:
        return 'Promotions'
    elif 'Updates' in output:
        return 'Updates'
    elif 'Priority' in output:
        return 'Priority'

    return output

In [ ]:
result1 = process_message(emails_df.iloc[0]['email_content'])
result2 = process_message(emails_df.iloc[1]['email_content'])

print('Email 1:', result1)
print('Email 2:', result2)

Expected output for the first two examples:

```text
Email 1: Priority
Email 2: Promotions
```